<a href="https://colab.research.google.com/github/sanmaykant/sec/blob/main/Notebooks/data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install edgartools beautifulsoup4 tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 23.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
BASE_PATH = "/content/drive/MyDrive/sec-risk-analyzer"
folders = [
    "data/raw",
    "data/processed"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_PATH, folder), exist_ok=True)
print("Folders created successfully!")

Folders created successfully!


In [4]:
from edgar import Company, set_identity
from tqdm import tqdm
import json

set_identity("analyst@edgarlytics.com")

In [5]:
def fetch_filing(ticker: str, year: int, form: str, section_name: str):
    company = Company(ticker)
    filings = company.get_filings(form=form)

    for filing in filings:
        try:
            filing_year = filing.filing_date.year

            if filing_year != year:
                continue

            parsed = filing.parse()
            if parsed is None:
                continue

            section = parsed.get_sec_section(section_name)

            if section is None:
                return {
                    "year": year,
                    "ticker": ticker,
                    "status": "missing_section"
                }

            return {
                "year": year,
                "ticker": ticker,
                "status": "success",
                "filing": section
            }

        except Exception as e:
            print(f"Error: {ticker} {year} → {e}")
            continue

    return {
        "year": year,
        "ticker": ticker,
        "status": "filing_not_found"
    }

In [6]:
tickers = ["AAPL", "META", "AMZN"]
years = [2024, 2023, 2022, 2021, 2020]

form = "10-K"
section_name = "part_i_item_1a"

results = []

for ticker in tickers:
    for year in tqdm(years, desc=f"Processing {ticker}"):
        res = fetch_filing(ticker, year, form, section_name)

        if res["status"] == "success":
            results.append(res)

Processing AMZN: 100%|██████████| 5/5 [00:20<00:00,  4.11s/it]


In [7]:
print("Valid filings:", len(results))
for r in results:
    print("\n---")
    print("Ticker:", r["ticker"])
    print("Year:", r["year"])
    print("Text length:", len(r.get("filing", "")))

Valid filings: 15

---
Ticker: AAPL
Year: 2024
Text length: 68887

---
Ticker: AAPL
Year: 2023
Text length: 67998

---
Ticker: AAPL
Year: 2022
Text length: 70911

---
Ticker: AAPL
Year: 2021
Text length: 66749

---
Ticker: AAPL
Year: 2020
Text length: 60994

---
Ticker: META
Year: 2024
Text length: 182525

---
Ticker: META
Year: 2023
Text length: 169104

---
Ticker: META
Year: 2022
Text length: 170031

---
Ticker: META
Year: 2021
Text length: 162073

---
Ticker: META
Year: 2020
Text length: 139874

---
Ticker: AMZN
Year: 2024
Text length: 59012

---
Ticker: AMZN
Year: 2023
Text length: 55596

---
Ticker: AMZN
Year: 2022
Text length: 52747

---
Ticker: AMZN
Year: 2021
Text length: 50069

---
Ticker: AMZN
Year: 2020
Text length: 49838


In [8]:
save_path = os.path.join(BASE_PATH, "data/processed/risk_data.json")
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved to: {save_path}")

Saved to: /content/drive/MyDrive/sec-risk-analyzer/data/processed/risk_data.json


In [9]:
sample = results[0]

print("Ticker:", sample["ticker"])
print("Year:", sample["year"])
print("\nSample text:\n")
print(sample["filing"][:1000])

Ticker: AAPL
Year: 2024

Sample text:

Item 1A.    Risk Factors

The Company’s business, reputation, results of operations, financial condition and stock price can be affected by a number of factors, whether currently known or unknown, including those described below. When any one or more of these risks materialize from time to time, the Company’s business, reputation, results of operations, financial condition and stock price can be materially and adversely affected.

Because of the following factors, as well as other factors affecting the Company’s results of operations and financial condition, past financial performance should not be considered to be a reliable indicator of future performance, and investors should not use historical trends to anticipate results or trends in future periods. This discussion of risk factors contains forward-looking statements.

This section should be read in conjunction with Part II, Item 7, “Management’s Discussion and Analysis of Financial Condition 